# Integración · Los *skills* académicos en ARAS

**Complemento de los Módulos 5 y 6.** ARAS reutiliza la **suite de investigación
académica** ya existente (abstract-writer, reviewer, paper-outline,
title-generator, etc.) en lugar de reescribirla.

## Objetivos

- Descubrir los **skills** instalados y convertirlos en capacidades de ARAS.
- Exponerlos como **herramientas MCP** para que el agente los descubra e invoque.
- Mapearlos a **roles** del equipo multiagente.

## Las tres vías de integración

1. **Como prompts:** el contenido de cada `SKILL.md` se inyecta como *system prompt*.
2. **Como herramientas MCP:** un servidor expone un *tool* por skill (Módulo 5).
3. **Como roles multiagente:** cada skill se asigna a un rol del equipo (Módulo 6).

## 1. Dependencias

In [ ]:
%pip install -q ollama mcp langchain-core langchain-ollama langgraph pydantic

## 2. Localizar el código del proyecto e importar ARAS

El código vive en `src/aras/`. Añadimos esa carpeta a `sys.path` (el kernel de
Jupyter se ejecuta desde la carpeta `notebooks/`).

In [ ]:
import sys, os
from pathlib import Path

def encontrar_src():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / 'src' / 'aras').exists():
            return str(base / 'src')
    return None

SRC = encontrar_src() or os.path.abspath('../src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print('src en sys.path:', SRC)

import aras
from aras.skills import get_registry, ejecutar_skill, RESEARCH_ROLES
print('ARAS versión:', aras.__version__)

## 3. El registro de skills

`SkillRegistry` descubre los `SKILL.md` en `~/.pi/agent/skills`, parsea su
frontmatter (`name`, `description`) y su cuerpo de instrucciones.

In [ ]:
reg = get_registry()
print('Skills totales descubiertos:', len(reg))
print('Skills de la suite académica:', len(reg.skills(academic_only=True)))
print()
for s in reg.skills(academic_only=True):
    print(f'  - {s.name:24s} {s.title}')

## 4. Ejecutar un skill (modo `dry_run`)

`ejecutar_skill` compone el prompt (instrucciones del skill + entrada). En modo
`dry_run` no llama al LLM: solo muestra el prompt construido, útil para depurar.

In [ ]:
out = ejecutar_skill(
    'title-generator',
    'Un artículo que propone un sistema RAG para revisiones sistemáticas.',
    dry_run=True,
)
print(out[:800])
print('...')

## 5. Ejecutar un skill (con el LLM local)

Requiere **Ollama corriendo** (`ollama serve`) y el modelo descargado. Si no,
la función devuelve un mensaje de error claro en lugar de romper.

In [ ]:
resultado = ejecutar_skill(
    'title-generator',
    'Un artículo que propone un sistema RAG para revisiones sistemáticas.',
)
print(resultado)

## 6. Exponer los skills como herramientas MCP

Escribimos un *runner* que lanza el servidor MCP de skills (definido en
`src/aras/mcp/skill_server.py`). El servidor registra 2 herramientas genéricas
(`listar_skills`, `ejecutar_skill`) y **una herramienta por cada skill**
(`abstract_writer`, `title_generator`, `reviewer`, ...).

In [ ]:
%%writefile aras_skill_server.py
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', 'src'))
from aras.mcp.skill_server import server

if __name__ == '__main__':
    server.run()

## 7. Cliente MCP: descubrir los skills

In [ ]:
import asyncio, json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def demo_descubrimiento():
    params = StdioServerParameters(command='python', args=['aras_skill_server.py'])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            nombres = [t.name for t in tools.tools]
            print(f'Herramientas descubiertas: {len(nombres)}')
            print('Genéricas:', [n for n in nombres if n in ('listar_skills', 'ejecutar_skill')])
            print('Específicas:', [n for n in nombres if n not in ('listar_skills', 'ejecutar_skill')])
            print()
            r = await session.call_tool('listar_skills', {'academic_only': True})
            skills = json.loads(r.content[0].text)
            print('listar_skills devolvió', len(skills), 'skills. Ejemplo:')
            print(json.dumps(skills[0], ensure_ascii=False, indent=2))

await demo_descubrimiento()

## 8. Cliente MCP: invocar un skill concreto

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def demo_invocar():
    params = StdioServerParameters(command='python', args=['aras_skill_server.py'])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            r = await session.call_tool('title_generator', {'entrada': 'Un sistema RAG para revisiones sistemáticas'})
            print('Respuesta del skill title_generator:')
            print(r.content[0].text)

# Requiere Ollama corriendo; si no, el skill devuelve un mensaje de error claro.
await demo_invocar()

## 9. Agente que elige entre skills

Convertimos cada skill en una herramienta de LangChain y dejamos que un agente
`create_react_agent` decida cuál invocar según la petición del usuario.

In [ ]:
from langchain_core.tools import StructuredTool

def fabricar_tool(skill):
    def _run(entrada: str) -> str:
        return ejecutar_skill(skill.name, entrada)
    return StructuredTool.from_function(
        func=_run,
        name=f'skill_{skill.tool_name}',
        description=skill.description,
    )

herramientas_skills = [fabricar_tool(s) for s in reg.skills(academic_only=True)]
print('Herramientas de skills registradas:', len(herramientas_skills))

def ollama_disponible():
    try:
        import ollama
        ollama.list()
        return True
    except Exception:
        return False

if ollama_disponible():
    from langchain_ollama import ChatOllama
    from langgraph.prebuilt import create_react_agent

    llm = ChatOllama(model='llama3.2', temperature=0)
    agente = create_react_agent(llm, herramientas_skills)
    resultado = agente.invoke({
        'messages': [('user', 'Genera 3 títulos alternativos para un artículo sobre RAG en revisiones sistemáticas.')],
    })
    for m in resultado['messages']:
        print(f'[{type(m).__name__}]', getattr(m, 'content', ''))
else:
    print('Ollama no está corriendo. Inícialo (ollama serve) y vuelve a ejecutar esta celda.')

## 10. Skills como roles del equipo multiagente

`RESEARCH_ROLES` asigna cada skill a un rol del equipo (Módulo 6). Cada rol puede
implementarse como un nodo del grafo multiagente cuyo *system prompt* son las
instrucciones de sus skills.

In [ ]:
for rol, skills in RESEARCH_ROLES.items():
    print(f'{rol:14s} -> {", ".join(skills)}')

## Resumen

- Descubrimos los **20 skills académicos** con `SkillRegistry`.
- Los **ejecutamos** como prompts (dry-run y con LLM) vía `ejecutar_skill`.
- Los **expusimos como herramientas MCP** y los descubrimos/invocamos desde un cliente.
- Construimos un **agente** que selecciona el skill adecuado.
- Los **mapeamos a roles** del equipo multiagente.

**Nota:** algunos skills son orquestadores multi-paso (research-manager,
latex-template-filler, figure-designer). Como llamada única al LLM producen una
versión útil, pero su comportamiento completo requiere encadenar varios pasos
(lo que ARAS hace con el pipeline y el equipo multiagente).